# Stage 4 - Data Transformation

Reshapes the cleaned dataset into a compact, display-ready schema: **short column names**, **numeric money fields**, and **derived financial metrics**.

| # | Step |
| --- | --- |
| 1 | Rename long columns to short identifiers (`food_expenses_monthly` -> `food`) |
| 2 | Coerce money columns to numbers (midpoints of the range labels) |
| 3 | `Total Expenses` = Food + Transport + Academic + Other |
| 4 | `Savings` = Allowance - Total Expenses |
| 5 | `Budget Usage %` = (Total Expenses / Allowance) x 100 |

The logic lives in `src/transformer.py`. Run it with `python src/transformer.py`.

## Two naming layers

The brief asks for short Title Case headers **or** short `snake_case` internally. Both are used, on purpose:

* **`CANONICAL_COLUMNS`** (`food`, `total_expenses`, ...) - the internal contract. All transformation code addresses these.
* **`DISPLAY_NAMES`** (`Food`, `Total Expenses`, `Budget Usage %`, ...) - the Title Case labels written to the CSV.

Two required labels are **not valid Python identifiers** and need quoting in SQL:
`Work/Internship` (contains `/`) and `Budget Usage %` (contains a space and `%`). The canonical layer exists precisely so downstream code never has to quote them.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
if not (PROJECT_ROOT / "src" / "transformer.py").exists():
    PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import transformer as T

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 220)

print("project root :", PROJECT_ROOT)
print("stage 3 input:", T.CLEANED_DATA_PATH)

## 2. Load the cleaned data

Stage 3 normalises names and schema but **leaves the money fields as ordinal range labels** (`"1000 - 2000 \u20b9"`).
Converting them is Stage 4's job, so unlike the previous revision of this pipeline this step is **real work**, not a
pass-through.

In [ ]:
cleaned = T.load_cleaned_data()
print(f"shape: {cleaned.shape[0]} rows x {cleaned.shape[1]} columns")
cleaned[list(T.MONEY_SOURCE_COLUMNS)].head(5)

In [ ]:
range_like = cleaned[list(T.MONEY_SOURCE_COLUMNS)].astype(str).apply(
    lambda s: s.str.contains(r"[<>]|-|\u20b9", regex=True)
)
print("cells still holding a range label:", int(range_like.sum().sum()))
cleaned[list(T.MONEY_SOURCE_COLUMNS)].dtypes.value_counts()

## 3. Midpoint parser

The parser is idempotent - numbers pass straight through - and raises on anything unrecognised rather than silently
returning `0`, which would read as a genuine zero spend.

In [ ]:
examples = [
    "1000 - 2000 \u20b9",
    "4000 - 5000 \u20b9",
    "2000 - 3000 \u20b9",
    "500 - 1500 \u20b9",
    "< 1000 \u20b9",
    "< 500 \u20b9",
    "> 5000 \u20b9",
    "> 3000 \u20b9",
    "1,000 - 2,000 \u20b9",
    "1000 - 2000 \u00e2,\u00b9",
    1500,
    "not a number",
]
pd.DataFrame(
    {
        "input": [repr(v) for v in examples],
        "stripped": [T.strip_currency_artifacts(v) if isinstance(v, str) else v for v in examples],
        "midpoint": [T.parse_money_midpoint(v) for v in examples],
    }
)

> **Open-ended buckets.** The brief specifies `> 5000` -> `6250`, i.e. `1.25x` the bound, while `< 500` -> `250` is `0.5x`.
> Those two multipliers are deliberately **asymmetric**, and that asymmetry is a modelling choice, not a derivation:
> **148 of 1065 money cells (13.9%) are open-ended.** `6250` is the midpoint of an assumed `[5000, 7500]` interval, which
> the data does not support directly. If Stage 5 needs calibrated spend totals, revisit `OPEN_HIGH_UPPER_MULTIPLIER` -
> it is a single named constant for exactly that reason.

## 4. Transform

In [ ]:
t = T.transform_data(cleaned)

print(f"shape: {t.shape[0]} rows x {t.shape[1]} columns")
print("\ncanonical columns:")
for column in t.columns:
    print(f"  {column:16} {str(t[column].dtype):10} -> display {T.DISPLAY_NAMES[column]!r}")

In [ ]:
t.head(8)

## 5. Verify the derived metrics independently

Recomputed from the four components rather than read back from the frame.

In [ ]:
components = list(T.EXPENSE_COLUMNS)
expected_total = t[components].sum(axis=1)
expected_savings = t["allowance"] - expected_total
expected_usage = np.where(
    t["allowance"] != 0,
    (expected_total / t["allowance"] * 100).astype(float),
    np.nan,
)

assert (expected_total == t["total_expenses"]).all(), "Total Expenses"
assert (expected_savings == t["savings"]).all(), "Savings"
assert np.allclose(np.round(expected_usage, 1), t["budget_usage_pct"].astype(float), equal_nan=True), "Budget Usage %"
print("Total Expenses / Savings / Budget Usage % all verified\n")
pd.DataFrame(
    {
        "Total Expenses": t["total_expenses"],
        "Allowance": t["allowance"],
        "Savings": t["savings"],
        "Budget Usage %": t["budget_usage_pct"],
    }
).head(8)

### Negative savings and >100% usage are real, not bugs

All 213 respondents overspend their allowance, so `Savings` is negative and `Budget Usage %` exceeds 100 everywhere.
Both columns are deliberately **left uncapped** - clamping at zero or at 100% would erase the only signal they carry.
A zero allowance would yield `NA` rather than `inf`; the smallest allowance here is 250, so the guard is untripped.

In [ ]:
print(f"Savings        : {t['savings'].min()} .. {t['savings'].max()}  ({(t['savings'] < 0).sum()}/{len(t)} negative)")
print(f"Budget Usage % : {t['budget_usage_pct'].min():.1f} .. {t['budget_usage_pct'].max():.1f}  "
      f"({(t['budget_usage_pct'] > 100).sum()}/{len(t)} over 100%)")
print(f"infinite values: {int(np.isinf(t['budget_usage_pct'].astype(float)).sum())}")
t[["total_expenses", "allowance", "savings", "budget_usage_pct"]].describe().T

## 6. Header quality check

The brief asks specifically that headers be clean, short and properly capitalised.

In [ ]:
FUNCTION_WORDS = {"of", "and", "the", "a", "an", "in", "for", "on", "to"}

def is_capitalized(header):
    """Capitalised if each word starts uppercase, or is a lowercase function word.

    Naive ``str.title()`` is the wrong oracle here: it wants "Year Of Study" and
    "Upi App", both of which are worse. Acronyms (UPI) and function words (of)
    must be allowed to break the pattern.
    """
    for word in header.split():
        letters = [ch for ch in word if ch.isalpha()]
        if not letters:
            continue  # e.g. the "%" in "Budget Usage %"
        if word.lower() in FUNCTION_WORDS:
            continue
        if not letters[0].isupper():
            return False
    return True

display = T.apply_display_names(t)
headers = list(display.columns)

audit = pd.DataFrame(
    {
        "header": headers,
        "len": [len(h) for h in headers],
        "capitalized": [is_capitalized(h) for h in headers],
        "identifier_safe": [h.isidentifier() for h in headers],
    }
)
audit

In [ ]:
print("max header length      :", audit["len"].max())
print("not capitalized        :", audit.loc[~audit["capitalized"], "header"].tolist() or "none")
print("need SQL quoting       :", audit.loc[~audit["identifier_safe"], "header"].tolist() or "none")
print("uniqueness preserved   :", display.columns.is_unique)
print("longest header         :", audit.loc[audit["len"].idxmax(), "header"])

`Work/Internship` and `Budget Usage %` are the only two headers that are not identifier-safe. That is a consequence of the
requested labels, not a defect - which is why the internal schema keeps `snake_case`.

## 7. Invariants

In [ ]:
assert list(t.columns) == list(T.CANONICAL_COLUMNS), "unexpected column order"
assert t.shape == (213, 22), t.shape
assert (t[list(T.MONEY_COLUMNS)].dtypes == "Int64").all(), "money columns must be Int64"
assert t["savings"].dtype == "Int64" and t["budget_usage_pct"].dtype == "Float64"
assert t["total_expenses"].min() > 0 and t["allowance"].min() > 0
assert not np.isinf(t["budget_usage_pct"].astype(float)).any(), "infinite usage"
assert t["budget_usage_pct"].isna().sum() == 0
assert len(t) == len(cleaned), "row count changed"
assert t.index.is_unique and t.index.is_monotonic_increasing
for h in display.columns:
    assert h == h.strip() and "  " not in h, f"messy header {h!r}"

# the stage must be re-runnable on its own output
again = T.transform_data(t)
numeric = [c for c in T.CANONICAL_COLUMNS if pd.api.types.is_numeric_dtype(t[c])]
assert all(t[c].equals(again[c]) for c in numeric), "not idempotent"

print("all invariants passed")

## 8. Export

In [ ]:
destination = T.export_transformed_data(t)
print("wrote:", destination)

roundtrip = pd.read_csv(destination)
assert roundtrip.shape == (213, 22), roundtrip.shape
assert list(roundtrip.columns) == [T.DISPLAY_NAMES[c] for c in T.CANONICAL_COLUMNS]
print(f"verified on disk: {roundtrip.shape[0]} rows x {roundtrip.shape[1]} columns")
print("\nheaders as written:")
print(",".join(roundtrip.columns))
roundtrip.head(3)

## Handoff notes for Stage 5

1. **All money fields are now numeric, and the open-ended rule lives only here.** `transformer.py` is self-contained and
   does not import from `cleaner.py`, so `OPEN_HIGH_UPPER_MULTIPLIER` is the single place `> X` is interpreted. Do not
   re-introduce a second copy in Stage 3 or the 13.9% of open-ended cells will diverge silently.
2. **`Budget Usage %` is a ratio, not a share** - it is not centred on 100%, so a mean or sum is meaningless. Use the median.
3. **`Savings` and `Budget Usage %` are perfectly redundant** (`Savings = Allowance x (usage/100 - 1)`), so do not feed both
   into a regression without dropping one.
4. **Two CSV layers exist** - the canonical `snake_case` frame in memory, and the Title Case CSV on disk. Ten of the 22
   exported headers need SQL quoting; `Work/Internship` and `Budget Usage %` additionally are not valid identifiers.